In [ ]:
def evaluate_fusion_alert(global_flag, local_flags):
    """
    Evaluates fusion alert ruleset based on 1 global model binary and 6 local model binaries.
    
    Args:
        global_flag (bool/int): Indicates if the global model detected an anomaly (1 or True) or not (0 or False).
        local_flags (list): A list of 6 binary flags (bool/int) from local models (e.g., [0, 1, 0, 0, 0, 0]).
        
    Returns:
        dict: Contains 'alert' (int), 'likely_category' (str), and 'localized_to' (list).
    """
    any_local = any(local_flags)
    # Considering local_flags are positional for 6 stages
    stages_with_local_flags = [i for i, flag in enumerate(local_flags) if flag]
    
    if global_flag and any_local:
        return {
            "alert": 1,
            "likely_category": "corroborated (local+global agree)",
            "localized_to": stages_with_local_flags
        }
    elif global_flag and not any_local:
        return {
            "alert": 1,
            "likely_category": "possible multi-subsystem attack, OR local miss, OR global false alarm — needs review",
            "localized_to": []
        }
    elif not global_flag and any_local:
        return {
            "alert": 1,
            "likely_category": "possible single-subsystem attack, OR global miss",
            "localized_to": stages_with_local_flags
        }
    else:
        return {
            "alert": 0,
            "likely_category": "Normal",
            "localized_to": []
        }


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

# Load the CSV files (replace these paths with your actual file paths)
global_df = pd.read_csv('global_output.csv')
local_df = pd.read_csv('combined_output_simple.csv')
actual_df = pd.read_csv('actual_output.csv')

# Extract values (assuming correct column order as specified)
global_flags = global_df.iloc[:, 0].values
local_flags_matrix = local_df.iloc[:, :6].values
actual_flags = actual_df.iloc[:, 0].values

# Evaluate fusion logic
predicted_alerts = []
likely_categories = []

for i in range(len(global_flags)):
    g_flag = global_flags[i]
    l_flags = local_flags_matrix[i]
    
    result = evaluate_fusion_alert(g_flag, l_flags)
    predicted_alerts.append(result['alert'])
    likely_categories.append(result['likely_category'])

predicted_alerts = np.array(predicted_alerts)

# Print metrics
print("Confusion Matrix:")
print(confusion_matrix(actual_flags, predicted_alerts))

print("\nClassification Report:")
print(classification_report(actual_flags, predicted_alerts, zero_division=0))

print("\nMetrics Summary:")
print(f"Accuracy:  {accuracy_score(actual_flags, predicted_alerts):.4f}")
print(f"Precision: {precision_score(actual_flags, predicted_alerts, zero_division=0):.4f}")
print(f"Recall:    {recall_score(actual_flags, predicted_alerts, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(actual_flags, predicted_alerts, zero_division=0):.4f}")
